<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 06. word2vec: Skip-gram과 Negative Sampling — Distributed Representations of Words and Phrases and their Compositionality

- **원 논문:** [Distributed Representations of Words and Phrases and their Compositionality](https://arxiv.org/abs/1310.4546)
- **저자 / 발표:** Tomas Mikolov, Ilya Sutskever, Kai Chen, Greg Corrado, Jeffrey Dean · NeurIPS 2013 (2013)
- **난이도 / 예상 시간:** 중급 · 약 65분
- **선수 지식:** PyTorch 텐서, 임베딩, 이진 교차 엔트로피, 코사인 유사도

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

작은 문장 모음에서 Skip-gram 중심어-문맥 쌍을 만들고, unigram 분포의 3/4승으로 음성 표본을 뽑아 식 (4)의 SGNS 목적함수를 최적화한다. 손실 감소와 학습된 단어 공간의 코사인 구조를 확인한다.

**원 논문과 다른 것**

논문은 최대 약 10억 단어 규모 말뭉치와 큰 어휘, 다수의 음성 표본으로 단어·구 벡터 및 analogy 정확도를 평가했다. 여기서는 8개 미니 문장과 수십 개 파라미터만 사용하므로 원 논문의 품질 수치는 재현하지 않는다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §2, 식 (1): Skip-gram이 주변 단어의 로그확률을 최대화 | `make_skipgram_pairs`가 윈도 안의 (중심어, 문맥어) 양성 쌍을 생성 | 쌍 개수와 자기 자신이 문맥에 들어가지 않는다는 assert |
| §2.2, 식 (4): Negative Sampling 목적함수 | `sgns_loss`가 positive `log σ`와 negative `log σ(-·)`를 계산 | 스칼라·유한 손실 및 모든 임베딩에 대한 gradient assert |
| §2.2: noise distribution에 unigram의 3/4승 사용 | `noise_probs = count**0.75 / Z`의 noise distribution | 확률 합 1과 고빈도 단어 확률의 완화 여부를 검사 |
| §4: 학습된 단어·구 벡터의 의미적 규칙성 평가 | 학습 embedding의 코사인 유사도 행렬과 2차원 SVD 투영 | 손실 감소율, 유사도 범위, 시각화로 미니 재현을 진단 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
\mathcal{L}_{\mathrm{SGNS}}=-\log\sigma(v'_w{}^\top v_c)
-\sum_{j=1}^{K}\log\sigma(-v'_{n_j}{}^\top v_c)
$$

- $v_c$는 center, $v'_w$는 positive context, $v'_{n_j}$는 negative word embedding입니다.
- 전체 vocabulary softmax 대신 positive pair와 $K$개 negative pair를 binary 분류합니다.
- window pair 생성, unigram negative sampling, embedding 점수, loss를 따로 구현합니다.
- index $[B]$는 embedding $[B,D]$로 변하고 dot product는 logit $[B]$를 만듭니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §2, 식 (1): Skip-gram이 주변 단어의 로그확률을 최대화
- **노트북 구현:** `make_skipgram_pairs`가 윈도 안의 (중심어, 문맥어) 양성 쌍을 생성
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** 쌍 개수와 자기 자신이 문맥에 들어가지 않는다는 assert를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 논문은 최대 약 10억 단어 규모 말뭉치와 큰 어휘, 다수의 음성 표본으로 단어·구 벡터 및 analogy 정확도를 평가했다. 여기서는 8개 미니 문장과 수십 개 파라미터만 사용하므로 원 논문의 품질 수치는 재현하지 않는다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 핵심 아이디어

Skip-gram은 중심 단어 $w_I$로 주변 단어 $w_O$를 예측한다. 전체 어휘
softmax 대신 Negative Sampling은 실제 쌍에는
$\log\sigma(v'_{w_O}{}^\top v_{w_I})$, 잡음 단어 $w_i$에는
$\log\sigma(-v'_{w_i}{}^\top v_{w_I})$를 준다(논문 식 (4)).

아래 실습에서는 논문이 보고한 `unigram_frequency ** 0.75` 잡음 분포도
그대로 구현한다. 작은 데이터이므로 최종 analogy 점수보다 목적함수와
임베딩 기하가 올바르게 연결되는지가 핵심이다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** center/context ids [B] → embeddings [B,D] → positive/negative logits [B]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import math
from collections import Counter

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

SEED = 606
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
print(f"torch={torch.__version__}, seed={SEED}; {ACCELERATOR.summary()}")

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2, 식 (1): Skip-gram이 주변 단어의 로그확률을 최대화 / §2.2: noise distribution에 unigram의 3/4승 사용
- **이 셀의 책임:** `make_skipgram_pairs`가 윈도 안의 (중심어, 문맥어) 양성 쌍을 생성 / `noise_probs = count**0.75 / Z`의 noise distribution
- **Tensor shape 계약:** center/context ids [B] → embeddings [B,D] → positive/negative logits [B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 쌍 개수와 자기 자신이 문맥에 들어가지 않는다는 assert / 확률 합 1과 고빈도 단어 확률의 완화 여부를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.tensor,torch.float32,Tensor.sum,ACCELERATOR.move,torch.isclose,Tensor.argmax -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 6개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.float32</code></summary>

#### `torch.float32`

- **역할:** 32비트 부동소수점 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** 일반적인 학습 실수 dtype입니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.float32](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

<details>
<summary><code>torch.isclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.isclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 텐서 원소마다 허용 오차 이내인지 검사합니다.
- **입력·반환:** 두 텐서를 받고 원소별 불리언 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 오차가 발생한 위치를 텐서 단위로 진단합니다.
- **주의점:** 전체 성공 여부가 필요하면 결과를 모두 축약하거나 `allclose`를 사용합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.isclose](https://docs.pytorch.org/docs/stable/generated/torch.isclose.html)

</details>

<details>
<summary><code>tensor.argmax(dim=None, keepdim=False)</code></summary>

#### `tensor.argmax(dim=None, keepdim=False)`

- **역할:** 지정 축에서 최댓값의 index를 찾습니다.
- **입력·반환:** 텐서와 축을 받고 정수 index 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** logit을 예측 class나 선택 action으로 바꿉니다.
- **주의점:** 미분 불가능하며 동점에서는 첫 index가 선택됩니다.

##### 관련 수식과 코드 연결

$$
k^\star=\operatorname*{arg\,max}_{k}x_k
$$

- **기호:** $k^\star$는 가장 큰 값 자체가 아니라 그 값의 index입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 어느 후보 집합에서 최댓값 index를 고를지 정합니다.
- **직관:** 연속 score를 이산 class·action 결정으로 바꾸는 선택 연산입니다.
- **shape 확인:** 선택 축이 제거되며 미분 가능한 gradient 경로는 만들지 않습니다.
- **공식 문서:** [Tensor.argmax](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
sentences = [
    "king royal palace queen royal palace".split(),
    "queen royal crown king royal crown".split(),
    "man human adult woman human adult".split(),
    "woman human person man human person".split(),
    "paris france capital berlin germany capital".split(),
    "berlin germany capital paris france capital".split(),
    "cat pet animal dog pet animal".split(),
    "dog pet animal cat pet animal".split(),
]
vocab = sorted({token for sentence in sentences for token in sentence})
stoi = {token: index for index, token in enumerate(vocab)}


def make_skipgram_pairs(rows, window=2):
    result = []
    for row in rows:
        ids = [stoi[token] for token in row]
        for center_position, center_id in enumerate(ids):
            left = max(0, center_position - window)
            right = min(len(ids), center_position + window + 1)
            for context_position in range(left, right):
                if context_position != center_position:
                    result.append((center_id, ids[context_position]))
    return torch.tensor(result, dtype=torch.long)


pairs = make_skipgram_pairs(sentences, window=2)
token_counts = Counter(token for row in sentences for token in row)
counts = torch.tensor([token_counts[word] for word in vocab], dtype=torch.float32)
noise_probs = counts.pow(0.75)
noise_probs = noise_probs / noise_probs.sum()
pairs, counts, noise_probs = ACCELERATOR.move(pairs, counts, noise_probs)

assert pairs.ndim == 2 and pairs.shape[1] == 2
assert torch.all(pairs[:, 0] != pairs[:, 1])
assert torch.isclose(noise_probs.sum(), noise_probs.new_tensor(1.0))
most_frequent = int(counts.argmax())
assert noise_probs[most_frequent] < counts[most_frequent] / counts.sum()
print(f"vocab={len(vocab)}, positive_pairs={len(pairs)}")

### 구현 단계 3 · 목적함수·학습 update

- **원문 위치:** §2.2, 식 (4): Negative Sampling 목적함수
- **이 셀의 책임:** `sgns_loss`가 positive `log σ`와 negative `log σ(-·)`를 계산
- **Tensor shape 계약:** center/context ids [B] → embeddings [B,D] → positive/negative logits [B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 스칼라·유한 손실 및 모든 임베딩에 대한 gradient assert. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Embedding,torch.einsum,torch.zeros,Tensor.backward,torch.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Embedding(num_embeddings, embedding_dim, ...)</code></summary>

#### `nn.Embedding(num_embeddings, embedding_dim, ...)`

- **역할:** 정수 ID를 학습 가능한 벡터로 조회합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
e_i=E[i],\qquad E\in\mathbb{R}^{V\times D}
$$

- **기호:** $i$는 정수 ID, $V$는 vocabulary 크기, $D$는 embedding 차원입니다.
- **수식 → 코드:** forward는 one-hot 행렬곱을 만들지 않고 weight 행 $E[i]$를 직접 조회합니다.
- **직관:** 범주 ID를 학습 가능한 연속 벡터 좌표로 바꿉니다.
- **shape 확인:** 입력 `[...] -> 출력 [..., D]`; 입력 dtype은 보통 `torch.int64`입니다.
- **공식 문서:** [nn.Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

</details>

<details>
<summary><code>torch.einsum(equation, *operands)</code></summary>

#### `torch.einsum(equation, *operands)`

- **역할:** 축 표기로 텐서 연산을 정의합니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
C_{ij}=\sum_{k=1}^{K}A_{ik}B_{kj}
$$

- **기호:** $k$는 곱한 뒤 사라지는 축이고 $i,j$는 결과에 남는 축입니다.
- **수식 → 코드:** `matmul`은 마지막 두 축에 이 합을 적용하고, `bmm`은 batch별로, `einsum`은 문자열로 축 관계를 지정합니다.
- **직관:** 공통 축 $k$의 정보를 가중합해 두 표현 사이의 상호작용을 만듭니다.
- **shape 확인:** `[..., I, K] @ [..., K, J] -> [..., I, J]`이며 앞쪽 batch 축은 broadcast될 수 있습니다.
- **공식 문서:** [torch.einsum](https://docs.pytorch.org/docs/stable/generated/torch.einsum.html)

</details>

<details>
<summary><code>torch.zeros(*size, dtype=None, device=None)</code></summary>

#### `torch.zeros(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 0인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 상태·mask·accumulator를 명시적으로 준비합니다.
- **주의점:** dtype과 device를 생략하면 주변 텐서와 달라질 수 있습니다.
- **공식 문서:** [torch.zeros](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>torch.isfinite(input)</code></summary>

#### `torch.isfinite(input)`

- **역할:** 텐서의 각 원소가 유한수인지 검사합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 불리언 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 gradient의 NaN·무한대 발생을 빠르게 찾습니다.
- **주의점:** 검사 결과는 미분 대상이 아니며 원인 자체를 고치지는 않습니다.
- **공식 문서:** [torch.isfinite](https://docs.pytorch.org/docs/stable/generated/torch.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class SGNS(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.input_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.output_embeddings = nn.Embedding(vocab_size, embedding_dim)
        bound = 0.5 / embedding_dim
        nn.init.uniform_(self.input_embeddings.weight, -bound, bound)
        nn.init.zeros_(self.output_embeddings.weight)

    def forward(self, center_ids, positive_ids, negative_ids):
        center = self.input_embeddings(center_ids)  # [P, D]
        positive = self.output_embeddings(positive_ids)  # [P, D]
        negatives = self.output_embeddings(negative_ids)  # [P, K, D]
        positive_scores = (center * positive).sum(dim=-1)  # [P]
        negative_scores = torch.einsum("pd,pkd->pk", center, negatives)
        return positive_scores, negative_scores


def sgns_loss(positive_scores, negative_scores):
    positive_term = F.logsigmoid(positive_scores)
    negative_term = F.logsigmoid(-negative_scores).sum(dim=1)
    return -(positive_term + negative_term).mean()


probe = SGNS(len(vocab), embedding_dim=12).to(DEVICE)
probe_negatives = torch.zeros((4, 3), dtype=torch.long, device=DEVICE)
probe_positive, probe_negative = probe(pairs[:4, 0], pairs[:4, 1], probe_negatives)
probe_loss = sgns_loss(probe_positive, probe_negative)
probe_loss.backward()
assert probe_positive.shape == (4,) and probe_negative.shape == (4, 3)
assert probe_loss.ndim == 0 and torch.isfinite(probe_loss)
assert probe.input_embeddings.weight.grad is not None
assert probe.output_embeddings.weight.grad is not None

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §2.2, 식 (4): Negative Sampling 목적함수 / §2.2: noise distribution에 unigram의 3/4승 사용
- **이 셀의 책임:** `sgns_loss`가 positive `log σ`와 negative `log σ(-·)`를 계산 / `noise_probs = count**0.75 / Z`의 noise distribution
- **Tensor shape 계약:** center/context ids [B] → embeddings [B,D] → positive/negative logits [B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 스칼라·유한 손실 및 모든 임베딩에 대한 gradient assert / 확률 합 1과 고빈도 단어 확률의 완화 여부를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.multinomial,Tensor.detach,torch.optim.Adam,Module.parameters,Optimizer.zero_grad,Optimizer.step -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 6개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.multinomial(input, num_samples, replacement=False, ...)</code></summary>

#### `torch.multinomial(input, num_samples, replacement=False, ...)`

- **역할:** 각 행의 가중치에 비례해 범주 index를 샘플링합니다.
- **입력·반환:** 음이 아닌 가중치와 표본 수를 받고 정수 index 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 정책 action, negative sample, token을 확률적으로 선택합니다.
- **주의점:** 입력은 합이 1일 필요는 없지만 음수·NaN이 없어야 합니다.
- **공식 문서:** [torch.multinomial](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
torch.manual_seed(SEED)
model = SGNS(len(vocab), embedding_dim=12).to(DEVICE)
generator = torch.Generator().manual_seed(SEED + 1)
negative_ids = (
    torch.multinomial(
        noise_probs.detach().cpu(),
        num_samples=len(pairs) * 5,
        replacement=True,
        generator=generator,
    )
    .view(len(pairs), 5)
    .to(DEVICE)
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_history = []

for _ in range(160):
    optimizer.zero_grad()
    positive_scores, negative_scores = model(pairs[:, 0], pairs[:, 1], negative_ids)
    loss = sgns_loss(positive_scores, negative_scores)
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss.detach()))

reduction = 1.0 - loss_history[-1] / loss_history[0]
assert loss_history[-1] < 0.75 * loss_history[0]
assert math.isfinite(reduction)
print(
    f"initial_loss={loss_history[0]:.3f}, final_loss={loss_history[-1]:.3f}, "
    f"reduction={reduction:.1%}"
)

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §4: 학습된 단어·구 벡터의 의미적 규칙성 평가
- **이 셀의 책임:** 학습 embedding의 코사인 유사도 행렬과 2차원 SVD 투영
- **Tensor shape 계약:** center/context ids [B] → embeddings [B,D] → positive/negative logits [B]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 손실 감소율, 유사도 범위, 시각화로 미니 재현을 진단. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.inference_mode,F.normalize,Tensor.mean,torch.allclose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.inference_mode(mode=True)</code></summary>

#### `torch.inference_mode(mode=True)`

- **역할:** 추론 전용으로 autograd 관련 추적을 더 강하게 비활성화합니다.
- **입력·반환:** 불리언 모드를 받고 context manager 또는 decorator로 동작합니다.
- **이 셀에서 쓰는 이유:** 순수 평가 구간의 오버헤드를 줄입니다.
- **주의점:** 이 모드에서 만든 텐서는 이후 gradient 계산에 쓰기 어려울 수 있습니다.
- **공식 문서:** [torch.inference_mode](https://docs.pytorch.org/docs/stable/generated/torch.inference_mode.html)

</details>

<details>
<summary><code>F.normalize(input, p=2.0, dim=1, eps=1e-12)</code></summary>

#### `F.normalize(input, p=2.0, dim=1, eps=1e-12)`

- **역할:** 지정 축의 벡터 norm을 1로 정규화합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\hat x=\frac{x}{\max(\lVert x\rVert_p,\epsilon)}
$$

- **기호:** $p$는 norm 차수, $\epsilon$은 0으로 나누지 않기 위한 하한입니다.
- **수식 → 코드:** 코드의 `dim` 축에서 norm을 구한 뒤 모든 원소를 같은 값으로 나눕니다.
- **직관:** 벡터 크기를 제거해 방향이나 상대 패턴에 집중하게 합니다.
- **shape 확인:** 입력과 출력 shape은 같고 선택 축의 norm은 약 1입니다.
- **공식 문서:** [F.normalize](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.normalize.html)

</details>

<details>
<summary><code>tensor.mean(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.mean(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 산술평균을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch loss, feature 통계, metric을 하나의 대표값으로 축약합니다.
- **주의점:** 어느 축을 없애는지와 빈 텐서에서 NaN이 나올 수 있음을 확인합니다.

##### 관련 수식과 코드 연결

$$
\bar x=\frac{1}{N}\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택한 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 평균에 포함할 원소와 결과에서 사라질 축을 결정합니다.
- **직관:** 여러 관측값을 중심 위치 하나로 요약합니다.
- **shape 확인:** `keepdim=False`이면 축약한 축이 제거되고 참이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.mean](https://docs.pytorch.org/docs/stable/generated/torch.mean.html)

</details>

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
with torch.inference_mode():
    word_vectors = (model.input_embeddings.weight + model.output_embeddings.weight) / 2
    word_vectors = F.normalize(word_vectors, dim=1)
    focus_words = ["king", "queen", "man", "woman", "cat", "dog"]
    focus_ids = torch.tensor([stoi[word] for word in focus_words], device=DEVICE)
    focus_vectors = word_vectors[focus_ids]
    cosine_matrix = focus_vectors @ focus_vectors.T
    centered = focus_vectors - focus_vectors.mean(dim=0, keepdim=True)
    _, _, vh = torch.linalg.svd(centered, full_matrices=False)
    coordinates = centered @ vh[:2].T

assert cosine_matrix.shape == (len(focus_words), len(focus_words))
assert torch.allclose(
    cosine_matrix.diag(),
    cosine_matrix.new_ones(len(focus_words)),
    atol=1e-5,
)
assert torch.all((-1.0001 <= cosine_matrix) & (cosine_matrix <= 1.0001))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(loss_history)
axes[0].set(title="SGNS objective", xlabel="step", ylabel="loss")
axes[1].scatter(coordinates[:, 0].detach().cpu(), coordinates[:, 1].detach().cpu())
for word, (x, y) in zip(focus_words, coordinates.tolist()):
    axes[1].annotate(word, (x, y), xytext=(4, 4), textcoords="offset points")
axes[1].set_title("learned word vectors (SVD projection)")
plt.tight_layout()
plt.show()

## 해석 체크

- 왜 출력 임베딩이 입력 임베딩과 별도로 필요한가?
- `0.75`를 `1.0`으로 바꾸면 자주 등장하는 단어가 음성 표본에서 어떻게 달라지는가?
- 이 작은 투영이 원 논문의 analogy 정확도를 재현했다고 말할 수 없는 이유를 적어 보세요.

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `논문은 최대 약 10억 단어 규모 말뭉치와 큰 어휘, 다수의 음성 표본으로 단어·구 벡터 및 analogy 정확도를 평가했다. 여기서는 8개 미니 문장과 수십 개 파라미터만 사용하므로 원 논문의 품질 수치는 재현하지 않는다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.